In [4]:
# SET UP 
import wandb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
import unicodedata

device = "cpu"

TRAINING_DATA = "~/Downloads/training_data_clean.csv"

# def prep_data():
df = pd.read_csv(TRAINING_DATA)

# rename
df.columns = [
    'id',
    'best_tasks_free',
    'acad_tasks_rating',
    'best_tasks_select',
    'subopt_freq_rating',
    'subopt_tasks_select',
    'subopt_tasks_free',
    'evidence_freq_rating',
    'verify_freq_rating',
    'verify_method_free',
    'target'
    ]

text_cols = [
    'best_tasks_free',
    'subopt_tasks_free',
    'verify_method_free',
    ]


ord_cols = [
    'acad_tasks_rating',
    'subopt_freq_rating',
    'evidence_freq_rating',
    'verify_freq_rating',
    ]

likert_categories = [
    '1 — Not at all likely',
    '2 — Unlikely',
    '3 — Neutral / Unsure',
    '4 — Likely',
    '5 — Very likely'
]

freq_categories = [
    '1 — Never',
    '2 — Rarely',
    '3 — Sometimes',
    '4 — Often',
    '5 — Very often'
]

ord_categories = [
    likert_categories,  # acad_tasks_rating
    freq_categories,    # subopt_freq_rating
    freq_categories,    # evidence_freq_rating
    freq_categories,    # verify_freq_rating
]

cat_cols = [
    'best_tasks_select',
    'subopt_tasks_select',
    ]

cat_multi_select_choices = [
    "Explaining complex concepts simply",
    "Drafting professional text (e.g., emails, résumés)",
    "Brainstorming or generating creative ideas",
    "Writing or editing essays/reports",
    "Converting content between formats (e.g., LaTeX)",
    "Writing or debugging code",
    "Data processing or analysis",
    "Math computations",
]

In [6]:
# split 
def split_data(df):
    students = df['id'].unique()
    train_idx, test_idx = train_test_split(
        students, test_size=0.3, random_state=22
    )

    train_df = df[df['id'].isin(train_idx)]
    test_df = df[df['id'].isin(test_idx)]

    train_groups = train_df['id'].values
    test_groups = test_df['id'].values

    x_train = train_df.drop(columns=['target', 'id'])
    y_train = train_df['target']

    x_test = test_df.drop(columns=['target', 'id'])
    y_test = test_df['target']

    return x_train, x_test, y_train, y_test, train_groups, test_groups


In [7]:
# Preprocessing pipeline
# tfidf params are optimized params from Optuna 

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer, OrdinalEncoder
from typing import Dict, List
import unicodedata


text_cols = [
    'best_tasks_free',
    'subopt_tasks_free',
    'verify_method_free',
    ]


ord_cols = [
    'acad_tasks_rating',
    'subopt_freq_rating',
    'evidence_freq_rating',
    'verify_freq_rating',
    ]

likert_categories = [
    '1 — Not at all likely',
    '2 — Unlikely',
    '3 — Neutral / Unsure',
    '4 — Likely',
    '5 — Very likely'
]

freq_categories = [
    '1 — Never',
    '2 — Rarely',
    '3 — Sometimes',
    '4 — Often',
    '5 — Very often'
]

ord_categories = [
    likert_categories,  # acad_tasks_rating
    freq_categories,    # subopt_freq_rating
    freq_categories,    # evidence_freq_rating
    freq_categories,    # verify_freq_rating
]

cat_cols = [
    'best_tasks_select',
    'subopt_tasks_select',
    ]

cat_multi_select_choices = [
    "Explaining complex concepts simply",
    "Drafting professional text (e.g., emails, résumés)",
    "Brainstorming or generating creative ideas",
    "Writing or editing essays/reports",
    "Converting content between formats (e.g., LaTeX)",
    "Writing or debugging code",
    "Data processing or analysis",
    "Math computations",
]

# custom function makecorpus just to keep consistency in pipeline
class MakeCorpus(BaseEstimator, TransformerMixin):
    # required by TansformerMixin -ignore
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        # X is DataFrame with text columns
        return X.agg(' '.join, axis=1).tolist()

def preprocess_text():
    return Pipeline([
        ('imputer', SimpleImputer(strategy="constant", fill_value="")),
        ('combiner', MakeCorpus()),
        # adding optimized params from Optuna 
        ('tfidf', TfidfVectorizer(max_features=1000, 
                                  ngram_range=(1, 3), 
                                  min_df=2, 
                                  max_df=0.9130255379198275, 
                                  sublinear_tf=True, 
                                  norm='l2',
                                  use_idf=False))
    ])

def preprocess_ord():

    return make_pipeline(
        SimpleImputer(strategy="most_frequent"),
        OrdinalEncoder(categories=ord_categories)

    )

def _normalize(s: str) -> str:
    # normalize unicode, collapse spaces, strip
    s = unicodedata.normalize("NFKC", str(s))
    s = " ".join(s.split())
    return s.strip()

def _split_top_level_commas(s: str) -> List[str]:
    """
    Split on commas that are NOT inside parentheses.
    Example: "Drafting ... (e.g., emails, résumés), Math computations"
    -> ["Drafting ... (e.g., emails, résumés)", "Math computations"]
    """
    parts = []
    buf = []
    depth = 0
    for ch in s:
        if ch == '(':
            depth += 1
            buf.append(ch)
        elif ch == ')':
            depth = max(0, depth - 1)
            buf.append(ch)
        elif ch == ',' and depth == 0:
            parts.append("".join(buf))
            buf = []
        else:
            buf.append(ch)
    if buf:
        parts.append("".join(buf))
    return parts

class MultiSelectBinarizerPerColumn(BaseEstimator, TransformerMixin):
    """
    One-hot encode multi-select columns using a safe split and a fixed
    set of canonical choices. Clone-safe: __init__ does not modify params.
    If the original cell is NaN -> all dummies for that column are NaN.
    """
    def __init__(self, classes: List[str]):
        self.classes = classes  # DO NOT MODIFY HERE (clone-safe)

    # internal helpers use fitted attributes
    def _parse_cell(self, x) -> List[str]:
        if pd.isna(x) or (isinstance(x, str) and x.strip() == ""):
            return []
        norm = _normalize(x)
        pieces = [_normalize(p) for p in _split_top_level_commas(norm)]
        # keep only exact matches (normalized)
        return [p for p in pieces if p in self.classes_]

    def fit(self, X, y=None):
        X = pd.DataFrame(X)

        # create normalized, fitted classes (separate from init param)
        self.classes_ = [_normalize(c) for c in self.classes]

        self.mlbs_: Dict[str, MultiLabelBinarizer] = {}
        self.col_to_outcols_: Dict[str, List[str]] = {}

        for col in X.columns:
            mlb = MultiLabelBinarizer(classes=self.classes_)
            mlb.fit([[]])  # fit on fixed classes only
            self.mlbs_[col] = mlb
            self.col_to_outcols_[col] = [f"{col}__{c}" for c in mlb.classes_]
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        blocks = []
        for col in X.columns:
            mlb = self.mlbs_[col]
            outcols = self.col_to_outcols_[col]
            is_missing = X[col].isna()
            lists = X[col].apply(self._parse_cell)
            arr = mlb.transform(lists)
            df_bin = pd.DataFrame(arr, columns=outcols, index=X.index)
            df_bin.loc[is_missing, :] = np.nan  # preserve missingness for KNN later
            blocks.append(df_bin)
        return pd.concat(blocks, axis=1)

def preprocess_cat():
    return make_pipeline(MultiSelectBinarizerPerColumn(classes=cat_multi_select_choices),
                                 SimpleImputer(strategy="constant", fill_value=0))

def create_preprocessor():
    # create pipelines for each type, applied per column within each subset
    # pipelines run in tandem
    # changed names to shorter versions for param_grid referencing
    transformers = [
        ("text", preprocess_text(), text_cols), # pass in just the names of the columns for now, df specified later in full pipeline
        ("ord", preprocess_ord(), ord_cols),
         ("cat", preprocess_cat(), cat_cols),
    ]

    return ColumnTransformer(transformers=transformers)


In [8]:
# Pytorch Dataset Class
class MyDataset(Dataset):
    def __init__(self, X, y):
        # X is your preprocessed features (after ColumnTransformer)
        self.features = torch.FloatTensor(X.toarray())  # Convert sparse to dense, then to tensor
        self.labels = torch.LongTensor(y)  # Convert labels to tensor
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [9]:
# other helper functions for training, validation, early stopping - used in both fixed and tuned training
def validate_model(model, valid_dl, loss_func):
    model.eval() # turn off dropout, batchnorm 
    
    val_loss = 0.0
    num_correct = 0 
    
    with torch.inference_mode(): # doesn't compute gradients, just gives prediction
        
        for (features, labels) in valid_dl: 
            outputs = model(features)  # Exact forward pass to get predictions 
            val_loss += loss_func(outputs, labels) * labels.size(0)
            
            _, predicted = torch.max(outputs.data, 1)
            num_correct += (predicted == labels).sum().item()
    
    return val_loss / len(valid_dl.dataset), num_correct / len(valid_dl.dataset)

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.should_stop = False
        
    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

In [ ]:
# MAIN EXECUTION OF JUST TRAINING - INITIAL TEST 
# uses fixed hyperparameters, hardcoded below 

# DATA PREP =========================== 
x_train, x_test, y_train, y_test, train_groups, test_groups = split_data(df)

preprocessor = create_preprocessor()
x_train_processed = preprocessor.fit_transform(x_train)

label_encoder = LabelEncoder()
y_train_processed = label_encoder.fit_transform(y_train)

dataset = MyDataset(x_train_processed, y_train_processed)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

# test data 
x_test_processed = preprocessor.transform(x_test)

y_test_processed = label_encoder.transform(y_test)

testdataset = MyDataset(x_test_processed, y_test_processed)
test_loader = DataLoader(testdataset, batch_size=32, shuffle=False)

# MODEL  =========================== 
input_size = x_train_processed.shape[1]  # Number of features
model = nn.Sequential(
        nn.Linear(input_size, 256),     # input_size = number of features after preprocessing (~2000)
        nn.BatchNorm1d(256),            # Normalize the 256 values
        nn.ReLU(),                      # Activation function, max(0, x)                      
        nn.Dropout(0.5),               # Regularization
        # During training: ~128 active neurons (randomly selected each batch)
        # Each neuron learns to work independently to indentify patterns without other neurons 
        nn.Linear(256, 3))

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()

early_stopping = EarlyStopping(patience=3)

for epoch in range(50): # pass through dataset 50 times, 18 batches per pass 
    # train
    model.train()
    for features, labels in train_loader: # Get batch (32 samples)
        optimizer.zero_grad() # Reset gradients calculations themselves from last batch
        outputs = model(features)  # Forward pass
        loss = loss_func(outputs, labels)
        loss.backward() # Backpropagation
        optimizer.step() # Update weights based on gradients, these accumulated over batch
    
    # validate  
    val_loss, val_acc = validate_model(model, test_loader, loss_func)
    print(f"Epoch {epoch+1}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    # prevent overfitting 
    early_stopping(val_loss)
    if early_stopping.should_stop:
        print("Early stopping triggered")
        break

In [ ]:
import wandb
# MAIN EXECUTION OF TRAINING WITH WANDB, FOR HYPERPARAMETER TUNING
# reuses some functions from above, have to rebuild some with hyperparam options 

def build_dataset(dataset, batch_size, dataset_type):
    if dataset_type == 'train':
        return DataLoader(dataset, batch_size=batch_size, shuffle=True)
    elif dataset_type == 'test':
        return DataLoader(dataset, batch_size=batch_size, shuffle=False)

def build_network(input_size, hidden_units, dropout_rate):
    model = nn.Sequential(
        nn.Linear(input_size, hidden_units),     
        nn.BatchNorm1d(hidden_units),            
        nn.ReLU(),                      
        nn.Dropout(dropout_rate),               
        nn.Linear(hidden_units, 3),              
    ).to(device)
    return model

def build_optimizer(model, lr):
    return optim.Adam(model.parameters(), lr=lr)
    
def train(config=None):
    with wandb.init(config=config) as run:
        config = run.config # Pass sweep configuration with the hyperparameters to run experiment with.
        
        # DATA PREP =========================== 
        x_train, x_test, y_train, y_test, train_groups, test_groups = split_data(df)

        # training data 
        preprocessor = create_preprocessor()
        x_train_processed = preprocessor.fit_transform(x_train)

        label_encoder = LabelEncoder()
        y_train_processed = label_encoder.fit_transform(y_train)

        dataset = MyDataset(x_train_processed, y_train_processed)
        train_loader = build_dataset(dataset, config.batch_size, 'train')

        # test data 
        x_test_processed = preprocessor.transform(x_test)
        y_test_processed = label_encoder.transform(y_test)

        testdataset = MyDataset(x_test_processed, y_test_processed)
        test_loader = build_dataset(testdataset, config.batch_size, 'test')

        # MODEL  ===========================
        input_size = x_train_processed.shape[1]  # Number of features
        model = build_network(input_size, config.hidden_units, config.dropout_rate)

        optimizer = build_optimizer(model, config.lr)
        loss_func = nn.CrossEntropyLoss()

        early_stopping = EarlyStopping(patience=3)

        for epoch in range(config.epochs): 
            # train
            model.train()
            for features, labels in train_loader: # Get batch (32 samples)
                optimizer.zero_grad() # Reset gradients calculations themselves from last batch
                outputs = model(features)  # Forward pass
                loss = loss_func(outputs, labels)
                loss.backward() # Backpropagation
                optimizer.step() # Update weights based on gradients, these accumulated over batch
            
            # validate  
            val_loss, val_acc = validate_model(model, test_loader, loss_func)
            print(f"Epoch {epoch+1}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
            wandb.log({"val_loss": val_loss, "val_acc": val_acc})
            
            # prevent overfitting 
            early_stopping(val_loss)
            if early_stopping.should_stop:
                print("Early stopping triggered")
                break

# SWEEP CONFIG 
sweep_config = {
    'method': 'grid',
    'metric': {
        'name': 'val_acc', # neural net trained w gradient descent works better with minimize val loss 
        'goal': 'maximize'   
    },
    'parameters': {
        'lr': { # learning rate, how big a step we take during grad desc 
            'values': [0.1, 0.01, 0.001, 0.0001]
        },
        'batch_size': {
            'values': [16, 32, 64, 128]
        },
        'hidden_units': {
            'values': [128, 256, 512, 1024]
        },
        'dropout_rate': {
            'values': [0.3, 0.5, 0.7, 0.9]
        },
        'epochs': {
            'values': [20, 30, 40, 50]
        }
    }
}   
sweep_id = wandb.sweep(sweep_config, project='GenAI_Characterization')
wandb.agent(sweep_id, function=train) 

Create sweep with ID: 309s4vtc
Sweep URL: https://wandb.ai/caroline-lyu-university-of-toronto/GenAI_Characterization/sweeps/309s4vtc


wandb: Agent Starting Run: 55u0bx04 with config:
wandb: 	batch_size: 16
wandb: 	dropout_rate: 0.3
wandb: 	epochs: 20
wandb: 	hidden_units: 128
wandb: 	lr: 0.1


Epoch 1, Val Loss: 0.9054, Val Acc: 0.6024
Epoch 2, Val Loss: 0.7544, Val Acc: 0.6586
Epoch 3, Val Loss: 0.7753, Val Acc: 0.6707
Epoch 4, Val Loss: 0.7352, Val Acc: 0.6948
Epoch 5, Val Loss: 0.8308, Val Acc: 0.6546
Epoch 6, Val Loss: 0.8867, Val Acc: 0.6426
Epoch 7, Val Loss: 0.8961, Val Acc: 0.6466
Early stopping triggered


val_acc,▁▅▆█▅▄▄
val_loss,█▂▃▁▅▇█
val_acc,0.64659
val_loss,0.89606


wandb: Agent Starting Run: z9qqv9yk with config:
wandb: 	batch_size: 16
wandb: 	dropout_rate: 0.3
wandb: 	epochs: 20
wandb: 	hidden_units: 128
wandb: 	lr: 0.01


...

wandb: Agent Starting Run: 1djmlf5t with config:
wandb: 	batch_size: 16
wandb: 	dropout_rate: 0.3
wandb: 	epochs: 20
wandb: 	hidden_units: 512
wandb: 	lr: 0.1


Epoch 1, Val Loss: 1.4335, Val Acc: 0.6104
Epoch 2, Val Loss: 0.8359, Val Acc: 0.6185
Epoch 3, Val Loss: 0.7298, Val Acc: 0.6787
Epoch 4, Val Loss: 0.7265, Val Acc: 0.6948
Epoch 5, Val Loss: 0.7558, Val Acc: 0.6827
Epoch 6, Val Loss: 0.8424, Val Acc: 0.6627
Epoch 7, Val Loss: 0.8489, Val Acc: 0.6988
Early stopping triggered


val_acc,▁▂▆█▇▅█
val_loss,█▂▁▁▁▂▂
val_acc,0.6988
val_loss,0.84891


wandb: Agent Starting Run: 4nuo1gj4 with config:
wandb: 	batch_size: 16
wandb: 	dropout_rate: 0.3
wandb: 	epochs: 20
wandb: 	hidden_units: 512
wandb: 	lr: 0.01


Epoch 1, Val Loss: 0.7988, Val Acc: 0.6707
Epoch 2, Val Loss: 0.9111, Val Acc: 0.6586
Epoch 3, Val Loss: 0.9463, Val Acc: 0.6667
Epoch 4, Val Loss: 1.1237, Val Acc: 0.6546
Early stopping triggered
